# 02 - Träna och inferera

Det här notebooket bygger en riktig ML-pipeline med preprocessing, KMeans-clustering, modellval, sparad pipeline och inference på nya kunder.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data').exists() and (ROOT.parent.parent / 'data').exists():
    ROOT = ROOT.parent.parent

In [ ]:
data = pd.read_csv(ROOT / 'data' / 'processed' / 'customer_enriched.csv', parse_dates=['last_purchase_date'])
data.head()

In [ ]:
feature_cols_num = ['recency_days', 'frequency', 'monetary', 'avg_order_value', 'basket_size', 'avg_items_per_invoice', 'unique_products', 'avg_unit_price']
feature_cols_cat = ['RegionGroup']
X_df = data[feature_cols_num + feature_cols_cat].copy()
X_df.head()

In [ ]:
numerisk_pipeline = Pipeline([
    ('imputerare', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('skalare', StandardScaler()),
])

kategorisk_pipeline = Pipeline([
    ('imputerare', SimpleImputer(strategy='most_frequent')),
    ('kodare', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

forbehandlare = ColumnTransformer([
    ('num', numerisk_pipeline, feature_cols_num),
    ('cat', kategorisk_pipeline, feature_cols_cat),
])

X = forbehandlare.fit_transform(X_df)
X.shape

In [ ]:
# Jämför flera klusterantal. Silhouette-score är en vägledning, inte hela beslutet.
resultat = []
for k in range(2, 7):
    modell = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = modell.fit_predict(X)
    resultat.append({
        'k': k,
        'silhouette_score': silhouette_score(X, labels),
        'inertia': modell.inertia_,
    })
scores = pd.DataFrame(resultat)
scores

In [ ]:
sns.set_theme(style='whitegrid', context='talk')
fig, ax = plt.subplots(figsize=(10, 6))

sns.lineplot(
    data=scores,
    x='k',
    y='silhouette_score',
    marker='o',
    linewidth=2.5,
    color='#0b5cad',
    ax=ax,
)

bästa_rad = scores.loc[scores['silhouette_score'].idxmax()]
ax.scatter(bästa_rad['k'], bästa_rad['silhouette_score'], s=180, color='#d62728', zorder=5)
ax.axvline(bästa_rad['k'], color='#d62728', linestyle='--', linewidth=1, alpha=0.7)
ax.annotate(
    f"bäst k={int(bästa_rad['k'])} ({bästa_rad['silhouette_score']:.3f})",
    xy=(bästa_rad['k'], bästa_rad['silhouette_score']),
    xytext=(10, 18),
    textcoords='offset points',
    fontsize=11,
    color='#d62728',
    fontweight='bold',
)

for _, rad in scores.iterrows():
    ax.annotate(
        f"{rad['silhouette_score']:.3f}",
        (rad['k'], rad['silhouette_score']),
        textcoords='offset points',
        xytext=(0, 8),
        ha='center',
        fontsize=10,
        color='#333333',
    )

ax.set_title('Silhouette-score per antal kluster')
ax.set_xlabel('Antal kluster')
ax.set_ylabel('Silhouette-score')
ax.set_ylim(0, max(scores['silhouette_score']) + 0.04)
ax.set_xticks(scores['k'])
ax.text(
    0.01,
    0.02,
    'Högre är bättre, men inte alltid mest affärsmässigt relevant',
    transform=ax.transAxes,
    fontsize=10,
    color='#555555',
)

plt.tight_layout()
fig.savefig(ROOT / 'outputs' / 'figures' / 'silhouette_scores.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# k=4 ger ett bättre affärsfall än k=2 och är fortfarande tydligt separerat.
bästa_k = 4
pipeline = Pipeline([
    ('förbehandlare', forbehandlare),
    ('kmeans', KMeans(n_clusters=bästa_k, random_state=42, n_init=20)),
])

kluster = pipeline.fit_predict(X_df)
data['Cluster'] = kluster
data[['CustomerID', 'RegionGroup', 'recency_days', 'frequency', 'monetary', 'Cluster']].head()

In [ ]:
profil = data.groupby('Cluster').agg(
    CustomerCount=('CustomerID', 'count'),
    MeanRecency=('recency_days', 'mean'),
    MeanFrequency=('frequency', 'mean'),
    MeanMonetary=('monetary', 'mean'),
    MeanAOV=('avg_order_value', 'mean'),
    MeanBasketSize=('basket_size', 'mean'),
    MeanUniqueProducts=('unique_products', 'mean'),
).round(1)
profil

In [ ]:
hjältar = profil.sort_values(['MeanRecency', 'MeanMonetary']).index[0]
risk = profil.sort_values(['MeanRecency', 'MeanMonetary'], ascending=[False, True]).index[0]
storköp = profil.sort_values('MeanAOV', ascending=False).index[0]
remaining = [c for c in profil.index if c not in {hjältar, risk, storköp}]
regelbundna = remaining[0] if remaining else hjältar

cluster_name_map = {
    hjältar: 'Hjältar',
    storköp: 'Stora engångsköp',
    regelbundna: 'Regelbundna kunder',
    risk: 'Riskkunder med lågt värde',
}
if len(set(cluster_name_map.values())) < 4 and len(profil.index) >= 4:
    ordered = profil.sort_values('MeanMonetary', ascending=False).index.tolist()
    cluster_name_map = {
        ordered[0]: 'Hjältar',
        ordered[1]: 'Stora engångsköp',
        ordered[2]: 'Regelbundna kunder',
        ordered[3]: 'Riskkunder med lågt värde',
    }

data['ClusterName'] = data['Cluster'].map(cluster_name_map)

joblib.dump(pipeline, ROOT / 'models' / 'clustering_pipeline.joblib')
data.to_csv(ROOT / 'outputs' / 'clustered_customers.csv', index=False)
scores.to_csv(ROOT / 'outputs' / 'model_selection.csv', index=False)
profil.to_csv(ROOT / 'outputs' / 'cluster_profile.csv')
cluster_name_map

In [ ]:
sammanfattning = data.groupby('ClusterName').agg(
    customers=('CustomerID', 'count'),
    mean_recency=('recency_days', 'mean'),
    mean_monetary=('monetary', 'mean'),
    mean_frequency=('frequency', 'mean'),
).round(1)
sammanfattning

In [ ]:
nya_kunder = pd.read_csv(ROOT / 'data' / 'raw' / 'new_customers.csv')
regioner = pd.read_csv(ROOT / 'data' / 'regions.csv')
nya_enriched = nya_kunder.merge(regioner[['Country', 'RegionGroup']], on='Country', how='left')
prediktioner = pipeline.predict(nya_enriched[feature_cols_num + feature_cols_cat])
nya_enriched['PredictedCluster'] = prediktioner
nya_enriched['PredictedClusterName'] = nya_enriched['PredictedCluster'].map(cluster_name_map)
nya_enriched.to_csv(ROOT / 'outputs' / 'inference_results.csv', index=False)
nya_enriched[['CustomerID', 'Country', 'PredictedClusterName']]

Den sparade pipelinefilen i `models/clustering_pipeline.joblib` innehåller både preprocessing och klustring, så inference använder exakt samma steg som träningen.